# Latency Benchmark

Same pipeline as `FinalProject_metrics.ipynb` but **LLM baselines run locally via Ollama**

---

## Prerequisites

### 1. Hardware requirements

| Model | VRAM (4-bit) | Minimum GPU |
|---|---|---|
| Phi-4-mini (BASE / LoRA / LoRA+RAG) | ~3 GB | Any CUDA GPU |
| `qwen3:32b` | ~18 GB | RTX 3090 / RTX 4090 (24 GB) |
| `llama3.3:70b` | ~40 GB | A100 80 GB, or 2× 24 GB GPUs |

---

### 2. Install Ollama (one-time)

Download and install from **https://ollama.com** (Windows / macOS / Linux).

Verify:
```bash
ollama --version
```

---

### 3. Pull the LLM models (one-time, run in a terminal)

```bash
ollama pull qwen3:32b        # ~20 GB download
ollama pull llama3.3:70b    # ~40 GB download
```

Confirm both are ready:
```bash
ollama list
```

---

### 4. Start the Ollama server (keep running while the notebook executes)

```bash
ollama serve
```

> On **Windows**, the Ollama desktop app starts the server automatically after installation —
> you likely do not need to run `ollama serve` manually.
> The server listens on `http://localhost:11434`.

---

### 5. Run all cells top to bottom

Cell-11 will verify connectivity and assert that both LLM models are available before proceeding.

---

### Troubleshooting

| Error | Fix |
|---|---|
| `AssertionError: llama3.3:70b not found` | Run `ollama pull llama3.3:70b` and wait for download |
| `Connection refused` on cell-11 | Ollama server not running — start with `ollama serve` |
| CUDA out of memory on LLM inference | Set `LLM1_MODEL = "llama3.2:8b"` in cell-11 |
| SLM and LLM share GPU, LLM is slow | Close other GPU processes; both model groups use the same device |

In [1]:
# --- Google Colab (use when running in Colab) ---
# from google.colab import drive
# drive.mount("/content/drive")

# --- Local WSL2 (use when running locally) ---
# No drive mount needed; data is accessed via /mnt/c/...

In [2]:
# Changed: groq → openai (used as Ollama OpenAI-compatible client)
!pip -q install -U unsloth peft accelerate bitsandbytes sentence-transformers transformers chromadb evaluate sacrebleu rouge-score nltk bert_score openai

In [ ]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"


In [4]:
import json, re, time, warnings, logging
import numpy as np
import pandas as pd
import torch

# Suppress a transformers logging bug (bad % formatting in deprecation warning)
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SUBJECT = "fsd"   # "fsd" | "fcs" | "dma"

# --- Google Colab (use when running in Colab) ---
# BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor"

# --- Local WSL2 (use when running locally) ---
# BASE_DIR = "/mnt/c/Users/nastt/Documents/agentic-ai-tutor/backend/data"
BASE_DIR = "/data"

BASE_MODEL_NAME = "microsoft/Phi-4-mini-instruct"
QWEN_MODEL_NAME = "Qwen/Qwen3-32B"
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT = True

ADAPTER_DIR = f"{BASE_DIR}/adapters/lora_phi4mini_{SUBJECT.upper()}"
EVAL_JSONL_PATH = f"{BASE_DIR}/eval_qa/{SUBJECT}_eval_OK.jsonl"

# RAG (chunk index + chroma dir)
CHUNK_CSV_PATH = f"{BASE_DIR}/rag/{SUBJECT}/chunk_index.csv"
CHROMA_DIR = f"{BASE_DIR}/rag/{SUBJECT}/chroma_store"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 6
MAX_CONTEXT_TOKENS = 1400

SYSTEM = (
    "You are a helpful tutor. Answer clearly and step-by-step when appropriate. "
    "Output ONLY the answer to the question. "
    "Do NOT include repository names, metadata tags, or unrelated code. "
    "If code is required, output only minimal code relevant to the question."
)


In [5]:
import os

def load_eval_jsonl(path: str, limit: int | None = None, seed: int = 42) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            q = obj.get("question")
            a = obj.get("answer")  # reference answer
            if q and a:
                rows.append({"question": q, "reference": a})
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Eval file is empty or missing {question, answer} fields.")
    if limit is not None:
        df = df.sample(n=min(limit, len(df)), random_state=seed).reset_index(drop=True)
    return df

assert os.path.exists(EVAL_JSONL_PATH), f"Missing eval set: {EVAL_JSONL_PATH}"
df_eval = load_eval_jsonl(EVAL_JSONL_PATH, limit=None)
print("Eval size:", len(df_eval))
df_eval.head(3)


# Load SLM + LoRA

In [6]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel

# Base
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    attn_implementation="sdpa"
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(base_model)

# # LoRA model
lora_base, _ = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    attn_implementation="sdpa"
)
lora_model = PeftModel.from_pretrained(lora_base, ADAPTER_DIR)
FastLanguageModel.for_inference(lora_model)

print("Loaded base + lora.")

# Recommended for best accuracy at 4-bit:
QWEN_MODEL_NAME = "Qwen/Qwen3-32B"

qwen_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name=QWEN_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    attn_implementation="sdpa"
)

# Apply a chat template. For Qwen3, ChatML is commonly compatible.
qwen_tokenizer = get_chat_template(qwen_tokenizer, chat_template="chatml")

FastLanguageModel.for_inference(qwen_model)

# RAG retrieval (Chroma)

In [8]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

def safe_str(x) -> str:
    if x is None:
        return ""
    return x if isinstance(x, str) else str(x)

def normalize_text(s: str) -> str:
    s = safe_str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def approximate_count_tokens(text: str) -> int:
    text = safe_str(text)
    return 0 if not text else max(1, int(len(text) / 4))

def truncate_to_token_budget(hits, max_tokens: int):
    used = []
    parts = []
    total = 0
    for h in hits:
        t = int(h.get("tokens", approximate_count_tokens(h.get("text", ""))))
        if used and total + t > max_tokens:
            break
        parts.append(h["text"].strip())
        used.append(h)
        total += t
        if total >= max_tokens:
            break
    return "\n\n---\n\n".join([p for p in parts if p]), used

# Initialize ChromaDB client and collection once
os.makedirs(CHROMA_DIR, exist_ok=True)
_chroma_client = chromadb.PersistentClient(
    path=CHROMA_DIR,
    settings=Settings(anonymized_telemetry=False),
)
_chroma_collection = _chroma_client.get_or_create_collection(
    name="rag",
    metadata={"hnsw:space": "cosine"},
)

embedder = SentenceTransformer(EMBED_MODEL_NAME)

def retrieve_chunks(query: str, top_k: int = TOP_K):
    q = normalize_text(query)
    q_emb = embedder.encode([q], normalize_embeddings=True).astype(np.float32).tolist()[0]
    res = _chroma_collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]

    hits = []
    for doc, meta, dist in zip(docs, metas, dists):
        meta = meta or {}
        hits.append({
            "text": safe_str(doc),
            "meta": meta,
            "distance": float(dist) if dist is not None else None,
            "tokens": int(meta.get("chunk_tokens", approximate_count_tokens(doc))),
        })
    return hits

def build_rag_context(question: str, top_k: int = TOP_K, max_context_tokens: int = MAX_CONTEXT_TOKENS):
    hits = retrieve_chunks(question, top_k=top_k)
    context_text, used = truncate_to_token_budget(hits, max_context_tokens)
    return context_text, used

# Load LLM baselines (Ollama — local)

# Unified generation

In [9]:
STOP_MARKERS = [
    "<|im_end|>", "<|im_start|>",
    "<reponame>", "<gh_stars>",
]

def cut_garbage(text: str) -> str:
    text = (text or "").strip()
    for m in STOP_MARKERS:
        i = text.find(m)
        if i != -1:
            text = text[:i].strip()
            break
    return text

def gen_phi(model, question: str, context: str = "", max_new_tokens=192) -> tuple[str, int]:
    """Returns (text, n_generated_tokens)."""
    if context.strip():
        user_content = (
            "Use the following course material excerpts to answer.\n\n"
            f"COURSE MATERIAL:\n{context}\n\n"
            f"QUESTION:\n{question}"
        )
    else:
        user_content = question

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content},
    ]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    attn = torch.ones_like(input_ids)
    eos = tokenizer.eos_token_id
    pad = eos if tokenizer.pad_token_id is None else tokenizer.pad_token_id

    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos,
            pad_token_id=pad,
            use_cache=True,
        )

    gen_ids = out[0][input_ids.shape[1]:]
    n_tokens = int(gen_ids.shape[0])
    text = cut_garbage(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return text, n_tokens

def gen_qwen(model, tokenizer, question: str, context: str = "", max_new_tokens=192) -> tuple[str, int]:
    """Returns (text, n_generated_tokens). Deterministic and 'no thinking' oriented."""
    if context.strip():
        user_content = (
            "Use the following course material excerpts to answer.\n\n"
            f"COURSE MATERIAL:\n{context}\n\n"
            f"QUESTION:\n{question}"
        )
    else:
        user_content = question

    # Add an explicit constraint to avoid chain-of-thought / verbose reasoning
    # (Closest analogue to Ollama think=False)
    system_qwen = SYSTEM + " Do not reveal chain-of-thought. Give only the final answer."

    messages = [
        {"role": "system", "content": system_qwen},
        {"role": "user", "content": user_content},
    ]

    device = "cuda" if torch.cuda.is_available() else "cpu"

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    attn = torch.ones_like(input_ids)
    eos = tokenizer.eos_token_id
    pad = eos if tokenizer.pad_token_id is None else tokenizer.pad_token_id

    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos,
            pad_token_id=pad,
            use_cache=True,
        )

    gen_ids = out[0][input_ids.shape[1]:]
    n_tokens = int(gen_ids.shape[0])
    text = cut_garbage(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return text, n_tokens

def pred_base(q: str) -> tuple[str, int]:
    return gen_phi(base_model, q, context="")

def pred_lora(q: str) -> tuple[str, int]:
    return gen_phi(lora_model, q, context="")

def pred_llm2(q: str) -> tuple[str, int]:
    return gen_qwen(qwen_model, qwen_tokenizer, q, context="")

In [10]:
STOP_MARKERS = [
    "<|im_end|>", "<|im_start|>",
    "<reponame>", "<gh_stars>",
]

def cut_garbage(text: str) -> str:
    text = (text or "").strip()
    # Strip complete <think>...</think> block (Qwen3 chain-of-thought)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # Strip incomplete <think>... block (truncated by max_new_tokens)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)
    text = text.strip()
    for m in STOP_MARKERS:
        i = text.find(m)
        if i != -1:
            text = text[:i].strip()
            break
    return text

def gen(model, tokenizer, question: str, context: str = "", max_new_tokens=256) -> tuple[str, int]:
    """Returns (text, n_generated_tokens)."""
    if context.strip():
        user_content = (
            "Use the following course material excerpts to answer.\n\n"
            f"COURSE MATERIAL:\n{context}\n\n"
            f"QUESTION:\n{question}"
        )
    else:
        user_content = question

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids.to("cuda")
    else:
        input_ids = encoded.to("cuda")

    attn = torch.ones_like(input_ids)
    eos = tokenizer.eos_token_id
    pad = eos if tokenizer.pad_token_id is None else tokenizer.pad_token_id

    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos,
            pad_token_id=pad,
            use_cache=True,
        )

    gen_ids = out[0][input_ids.shape[1]:]
    n_tokens = int(gen_ids.shape[0])
    text = cut_garbage(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return text, n_tokens


# Run evaluation (SLM-base vs SLM-LoRA vs SLM-LoRA-RAG vs LLMs)

In [11]:
import torch

def run_eval(df: pd.DataFrame, limit: int | None = None) -> pd.DataFrame:
    recs = []
    n = len(df) if limit is None else min(limit, len(df))

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    for i in range(n):
        print("test " + str(i))
        q   = df.loc[i, "question"]
        ref = df.loc[i, "reference"]

        print("BASE started")
        # BASE
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        p_base, tok_base = pred_base(q)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        print("LoRA started")
        # LoRA
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t2 = time.perf_counter()
        p_lora, tok_lora = pred_lora(q)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t3 = time.perf_counter()

        print("LoRA+RAG started")
        # LoRA+RAG — retrieval and generation timed separately
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t4 = time.perf_counter()
        ctx, used_chunks = build_rag_context(q)
        t4b = time.perf_counter()                              # retrieval done
        p_lora_rag, tok_rag = gen_phi(lora_model, q, context=ctx)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t5 = time.perf_counter()

        print("QWEN started")
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t8 = time.perf_counter()
        p_llm2, tok_llm2 = pred_llm2(q)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t9 = time.perf_counter()

        lat_base    = t1 - t0
        lat_lora    = t3 - t2
        lat_rag_ret = t4b - t4
        lat_rag_gen = t5 - t4b
        lat_rag     = t5 - t4
        lat_llm2    = t9 - t8

        recs.append({
            "idx": i,
            "question": q,
            "reference": ref,
            "pred_base":     p_base,
            "pred_lora":     p_lora,
            "pred_lora_rag": p_lora_rag,
            "pred_llm2":     p_llm2,
            # latency
            "lat_base_s":          lat_base,
            "lat_lora_s":          lat_lora,
            "lat_lora_rag_s":      lat_rag,
            "lat_rag_retrieval_s": lat_rag_ret,
            "lat_rag_gen_s":       lat_rag_gen,
            "lat_llm2_s":          lat_llm2,
            # output token counts (for fair comparison verification)
            "n_tok_base":  tok_base,
            "n_tok_lora":  tok_lora,
            "n_tok_rag":   tok_rag,
            "n_tok_llm2":  tok_llm2,
            # tokens/sec (generation only, normalized per output token)
            "tps_base":  tok_base  / lat_base    if lat_base    > 0 else None,
            "tps_lora":  tok_lora  / lat_lora    if lat_lora    > 0 else None,
            "tps_rag":   tok_rag   / lat_rag_gen if lat_rag_gen > 0 else None,
            "tps_llm2":  tok_llm2  / lat_llm2   if lat_llm2    > 0 else None,
            # RAG metadata
            "rag_used_k":   len(used_chunks),
            "rag_avg_dist": float(np.mean([h.get("distance") for h in used_chunks
                                           if h.get("distance") is not None])) if used_chunks else None,
        })

        if (i + 1) % 10 == 0:
            print(f"Done {i+1}/{n}")

    return pd.DataFrame(recs)

df_out = run_eval(df_eval)
df_out.head(2)

# Summarize table

In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    configs = [
        ("BASE",            "pred_base",     "lat_base_s",     "tps_base",  "n_tok_base"),
        ("LoRA",            "pred_lora",     "lat_lora_s",     "tps_lora",  "n_tok_lora"),
        ("LoRA+RAG",        "pred_lora_rag", "lat_lora_rag_s", "tps_rag",   "n_tok_rag"),
        ("LLM (Qwen3-32B)", "pred_llm2",     "lat_llm2_s",     "tps_llm2",  "n_tok_llm2"),
    ]

    for model_name, pred_col, lat_col, tps_col, tok_col in configs:
        rows.append({
            "model":            model_name,
            "latency_mean_s":   float(df[lat_col].mean()),
            "latency_median_s": float(df[lat_col].median()),
            "tps_mean":         float(df[tps_col].mean())   if tps_col in df.columns else None,
            "tps_median":       float(df[tps_col].median()) if tps_col in df.columns else None,
            # avg output tokens — use to verify SLM vs LLM outputs are comparable length
            "n_tok_mean":       float(df[tok_col].mean())   if tok_col in df.columns else None,
        })

    return pd.DataFrame(rows)

df_summary = summarize(df_out)
df_summary

# Save CSV artifacts

In [ ]:
OUT_DIR = f"{BASE_DIR}/metrics_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Changed: filenames include "local" to distinguish from Groq-API run
pred_path = f"{OUT_DIR}/{SUBJECT}_preds_all_models_local.csv"
sum_path  = f"{OUT_DIR}/{SUBJECT}_metrics_all_models_local.csv"

df_out.to_csv(pred_path, index=False)
df_summary.to_csv(sum_path, index=False)

print("Saved:")
print(pred_path)
print(sum_path)

In [ ]:
# df_out["lat_base_s"]
float(df_out["lat_base_s"].mean())